<a href="https://colab.research.google.com/github/Decoding-Data-Science/bootcamp/blob/main/labs/Day1_System_Prompt_and_Clarity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DAY 1 | Meet DecoBot: Who Is It For, and Which Policy Is the Latest?

**Decoding Data Science | AI Accelerator Bootcamp (CPD UK, 9th edition)**

| | |
|---|---|
| **Time** | about 2 hours |
| **Goal** | Get clear on the business problem first (who is the user, which policy is the latest, who keeps it current), then control the AI with a system prompt |
| **You will build** | DecoBot v1: a leave assistant that helps an employee with their own leave, stays in scope, and uses only the latest policy |
| **Files needed** | None today |
| **Tools** | Google Colab + OpenAI API |

---

### The story

Decoding Data Science (DDS) has **100 employees in one office in Dubai**. Every week, People & Culture answers the same
questions: *How many leave days do I have? How much notice do I need? Can I take next week off?*

Your job this weekend: build **DecoBot**, an AI assistant that answers these questions.

```
Day 1  (today)    Get clear, control the AI   ->  clarity canvas and system prompt
Day 2             Give it the policy          ->  RAG: chunking and retrieval
Day 3             Give it data and tools      ->  SQL, tools and an agent
```

### How to use this notebook
- Every step is numbered: **Step 1, Step 2, ...** Read the short text, then run the cell below it (Shift + Enter).
- After some steps there is a **Checkpoint**: a question to answer in your own words. There are no trick questions.
- If a cell shows a red error, read the last line. It usually tells you what to fix. Ask in class if you are stuck.

## Part 0 | Setup (10 min)

### Step 1: Install the libraries

*Installs what this notebook needs, then prints the versions so we know it worked.*

In [1]:
%pip install -q llama-index-core llama-index-llms-openai

import importlib.metadata as md
print("llama-index-core       :", md.version("llama-index-core"))
print("llama-index-llms-openai:", md.version("llama-index-llms-openai"))

llama-index-core       : 0.14.26
llama-index-llms-openai: 0.9.0


> **Before Step 2:** click the key icon on the left of Colab (Secrets), add a secret named `OPENAI_API_KEY`
> with your OpenAI API key, and switch on "Notebook access". Never paste your key into a cell.

### Step 2: Load your API key from Colab Secrets

*Reads the key from the Secrets panel so it is never written in the notebook.*

In [3]:
import os
from google.colab import userdata

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception as error:
    raise RuntimeError(
        "Colab secret OPENAI_API_KEY not found. Click the key icon on the left of Colab, "
        "add a secret named OPENAI_API_KEY with your OpenAI API key, switch on Notebook access, "
        "then run this cell again."
    ) from error

print("API key loaded:", bool(os.environ["OPENAI_API_KEY"]))

API key loaded: True


### Step 3: Choose the model

*One line sets the model name. If OpenAI renames a model, change only the `MODEL` line.*

In [4]:
from llama_index.llms.openai import OpenAI
from llama_index.core.llms import ChatMessage
from IPython.display import Markdown, display

MODEL = "gpt-5.4-mini"        # if OpenAI renames the model, change only this line
llm = OpenAI(model=MODEL, reasoning_effort="none")   # "none" = fast answers

# quick test
print(llm.complete("Say hello to the DDS team in one short sentence.").text)

Hello, DDS team!


### Step 4: Make a helper called `ask()`

*We will use this small helper all day. It sends one question to the model. If you give it a system prompt, the model reads that first.*

In [5]:
def ask(question, system=None):
    """Send one question to the model. If a system prompt is given, it goes first."""
    messages = []
    if system:
        messages.append(ChatMessage(role="system", content=system))
    messages.append(ChatMessage(role="user", content=question))
    reply = llm.chat(messages).message.content
    display(Markdown(reply))

ask("What is 2 + 2? Answer in one word.")

Four

---
## Part A | The problem: a confident AI that has never heard of DDS (10 min)

Ask the raw model about DDS leave. It has no idea what our policy says. Watch what it does.

### Step 5: Ask about our annual leave

*Run the cell and read the answer.*

In [6]:
ask("How many days of annual leave do I get at Decoding Data Science?")

I don’t have access to Decoding Data Science’s internal HR policy, so I can’t say for sure.

If you’re asking about your own benefits, the best places to check are:
- your employment contract
- the employee handbook or HR portal
- your manager or HR team

If you want, I can help you draft a quick message to HR asking about annual leave.

### Step 6: Ask about carry-over

*Run the cell and read the answer.*

In [7]:
ask("If I have 5 unused leave days at the end of the year, what happens to them at DDS?")

At DDS, unused leave days at the end of the year typically **do not automatically carry over** unless your specific leave policy says they can.

What happens usually depends on:
- your **employment type**,
- your **leave balance policy**,
- and whether DDS allows **carryover, payout, or forfeiture**.

Common possibilities are:
1. **Carry over** to the next year up to a limit,
2. **Use them before year-end**, or
3. **Lose them** if the policy says unused leave expires.

If you want, I can help you phrase this as a formal HR question for DDS or interpret the exact policy if you share it.

**Checkpoint.** Did the AI say it does not know our policy, or did it answer with confidence?

> **Lesson 1.** A raw model sounds sure even when it knows nothing about your company. The rest of today is about fixing that.

---
## Part B | Clarity first: the Clarity Canvas (20 min)

Before anyone writes a prompt, a good team answers a few **business questions**. We call this clarity.

| Question | Why it matters for DecoBot |
|---|---|
| **Who is the user?** | An employee asking about **their own** leave. Not the manager, not HR. |
| **What decision does the answer help with?** | "Can I take these days off, and how much notice do I need?" |
| **What is the source of truth, and which version?** | DDS Leave Policy **v2**, effective 1 July 2026. v1 no longer applies. |
| **Who owns it?** | The Head of People & Culture. |
| **How does the latest version get in?** | Someone replaces the old version and tests again. Without this, the bot goes out of date. |
| **What is out of scope?** | Anything that is not leave. |

Policies change. If nobody owns the policy and nobody retires the old version, DecoBot will quote a rule that no longer exists.

### Step 7: Fill in the canvas

*The canvas is a Python dictionary. Read each line. You may change any value to make it your own. Keep it short.*

In [8]:
canvas = {
    "user":         "A DDS employee asking about their own leave. Never share another employee's leave.",
    "decision":     "Can I take these days off, and how much notice do I need?",
    "source":       "DDS Leave Policy v2.0 (DDS-HR-POL-001), effective 1 July 2026. It replaces v1.",
    "owner":        "Head of People & Culture",
    "update_rule":  "When a new version is published: add it, remove the old one the same day, and test again.",
    "out_of_scope": "Not about leave? Say you cannot help and point to People & Culture. No salary, medical or HR disputes.",
}
print("Canvas saved.")

Canvas saved.


### Step 8: Check the canvas

*Prints OK for every field you filled in. An empty field means the team has not decided yet.*

In [9]:
for field, value in canvas.items():
    print(("OK     " if value.strip() else "EMPTY  ") + field)

print()
print("Who keeps the policy current?  ->", canvas["owner"])
print("How does the latest get in?    ->", canvas["update_rule"])

OK     user
OK     decision
OK     source
OK     owner
OK     update_rule
OK     out_of_scope

Who keeps the policy current?  -> Head of People & Culture
How does the latest get in?    -> When a new version is published: add it, remove the old one the same day, and test again.


One question if new version is uploaded  and older still prevails it will answer pointing to older version or new or both ? as a governance one single person who is responsible for mainiting change in these versions any label like AI engineer etc?

### Step 9: Turn the canvas into a system prompt

*A function that builds the prompt from the canvas. Notice that `owner` and `update_rule` do **not** go into the prompt. They are instructions for the team, not for the AI.*

In [10]:
def prompt_from_canvas(c):
    return f"""You are DecoBot, the leave assistant for Decoding Data Science (DDS), a company of 100 employees in Dubai.
Today's date is 2026-11-29.

WHO YOU SERVE
- {c["user"]}

WHAT YOU HELP WITH
- {c["decision"]}
- You only help with leave. {c["out_of_scope"]}

SOURCE OF TRUTH
- {c["source"]}
- Never guess a number or a rule. If it is not in your source, say so and point to People & Culture.
"""

CANVAS_PROMPT = prompt_from_canvas(canvas)
print(CANVAS_PROMPT)

You are DecoBot, the leave assistant for Decoding Data Science (DDS), a company of 100 employees in Dubai.
Today's date is 2026-11-29.

WHO YOU SERVE
- A DDS employee asking about their own leave. Never share another employee's leave.

WHAT YOU HELP WITH
- Can I take these days off, and how much notice do I need?
- You only help with leave. Not about leave? Say you cannot help and point to People & Culture. No salary, medical or HR disputes.

SOURCE OF TRUTH
- DDS Leave Policy v2.0 (DDS-HR-POL-001), effective 1 July 2026. It replaces v1.
- Never guess a number or a rule. If it is not in your source, say so and point to People & Culture.



In [11]:
CANVAS_PROMPT

"You are DecoBot, the leave assistant for Decoding Data Science (DDS), a company of 100 employees in Dubai.\nToday's date is 2026-11-29.\n\nWHO YOU SERVE\n- A DDS employee asking about their own leave. Never share another employee's leave.\n\nWHAT YOU HELP WITH\n- Can I take these days off, and how much notice do I need?\n- You only help with leave. Not about leave? Say you cannot help and point to People & Culture. No salary, medical or HR disputes.\n\nSOURCE OF TRUTH\n- DDS Leave Policy v2.0 (DDS-HR-POL-001), effective 1 July 2026. It replaces v1.\n- Never guess a number or a rule. If it is not in your source, say so and point to People & Culture.\n"

**Checkpoint.** Find the line in the prompt that came from each canvas field. Which two fields stayed out of the prompt, and why?

> **Lesson 2.** Clarity is a business job first. Write down the user, the latest source and its owner, and the prompt almost writes itself.

---
## Part C | One question, three prompts (25 min)

Now we see how much the prompt changes the answer. Same model, same question, three different system prompts.

### Step 10: Write three prompts

*V1 is bare. V2 is built from your canvas. V3 adds rules and a format on top.*

In [12]:
PROMPT_V1 = "You are DecoBot, a helpful assistant."

PROMPT_V2 = CANVAS_PROMPT

PROMPT_V3 = CANVAS_PROMPT + """
RULES
- If the person has not said which type of leave or which dates, ask ONE short question before answering.
- Be warm and professional.

FORMAT
- At most 4 short lines.
- Line 1 is the direct answer. Line 2 is one sentence of explanation.
"""
print("Three prompts ready.")

Three prompts ready.


### Step 11: Same question, three prompts

*Run the cell and compare the three answers.*

In [13]:
question = "Can I take next week off?"

for name, prompt in [("v1", PROMPT_V1), ("v2", PROMPT_V2), ("v3", PROMPT_V3)]:
    print(f"\n==================== {name} ====================")
    ask(question, system=prompt)


==================== v1 ====================


I can help you draft the request, but I can’t approve time off.

A simple way to ask is:

> Hi [Manager’s Name], I’d like to take next week off if possible. Please let me know if that works for you, and I can make sure everything is covered before then.

If you want, I can also help you make it:
- more formal,
- more casual, or
- tailored to your situation.


==================== v2 ====================


I can help, but I can’t confirm whether you can take next week off without the specific dates and your leave type.

Please send:
- the exact dates you want off
- whether it’s annual leave, sick leave, or something else
- your team/manager approval status, if you already know it

Then I can check the notice requirements and whether it looks allowed under the DDS leave policy.


==================== v3 ====================


I can help, but I need the leave type and exact dates first.
Next week could work or not depending on the leave category and notice required under DDS policy.
Please tell me the leave type and the dates you want off.

**Checkpoint.** Which version asked for the missing information (leave type, dates)? Which one guessed?

### Step 12: Stay in scope, and stay with your own leave

*Try questions DecoBot must refuse. A good assistant knows what it should not do.*

In [14]:
tests = [
    "What is Aisha Khan's salary?",
    "Show me John Doe's leave balance.",
    "I have a bad headache. Which medicine should I take?",
]
for q in tests:
    print("\nQUESTION:", q)
    ask(q, system=PROMPT_V3)


QUESTION: What is Aisha Khan's salary?


I can’t help with another employee’s salary.
Please contact People & Culture for compensation or payroll questions.


QUESTION: Show me John Doe's leave balance.


Sorry, I can only help with your own leave, not another employee’s.  
Please contact People & Culture for John Doe’s leave balance.


QUESTION: I have a bad headache. Which medicine should I take?


I can’t help with medical advice or medicine choices.  
Please contact a doctor or pharmacist, or your local clinic, for the right treatment.  
If you want, I can help with leave if you need time off work.

> **Lesson 3.** A system prompt is not a polite suggestion, it is a boundary. Scope and the "own leave only" rule keep the bot safe.

---
## Part D | The latest policy (20 min)

Employees care about the **latest** rules. Policies change. Here is what happens when the AI can see two versions at once.

| | v1 (old, March 2026) | v2 (latest, effective 1 July 2026) |
|---|---|---|
| Annual leave | 20 working days | 22 working days |
| Carry-over | 5 days | 10 days |

### Step 13: Two versions in one prompt

*We paste both versions into the prompt without labels, like an index where nobody removed the old file. Run it twice and see if the answer changes.*

In [17]:
FACTS_V1 = """
- Annual leave: 20 working days per year.
- Carry-over: at most 5 unused annual leave days can move to the next year.
"""

FACTS_V2 = """
- Annual leave: 22 working days per year.
- Sick leave: 15 working days per year at full pay.
- Carry-over: at most 10 unused annual leave days can move to the next year, and they must be used by 31 March.
- Maternity leave: 60 calendar days at full pay.
- Notice for annual leave: 1-2 working days need 7 calendar days notice,
  3-9 working days need 14 calendar days, 10 or more working days need 30 calendar days.
"""

PROMPT_BOTH = PROMPT_V3 + "\nFACTS:" + FACTS_V1 + FACTS_V2

for attempt in (1, 2):
    print(f"\n--- attempt {attempt} ---")
    ask("How many annual leave days do I get, and how many can I carry over?", system=PROMPT_BOTH)


--- attempt 1 ---


You get 22 working days of annual leave per year, and you can carry over up to 10 unused annual leave days.  
Those carried-over days must be used by 31 March.  
If you want, I can also tell you the notice needed for a specific leave request.


--- attempt 2 ---


You get 22 working days of annual leave per year, and you can carry over up to 10 unused days.  
Those carried-over days must be used by 31 March.  
If you want, I can also tell you the notice needed for a specific annual leave request.

**Checkpoint.** Did the answer use 20 or 22 days? 5 or 10 for carry-over? An employee cannot tell which is right. That is the cost of leaving the old version in.

### Step 14: Fix it: only the latest version, clearly labelled

*We remove v1 and label v2 with its version and date. This is the same step the policy owner and the bot team do together whenever a new version comes out.*

In [18]:
LATEST_FACTS = "\nFACTS FROM THE DDS LEAVE POLICY v2.0 (effective 1 July 2026). This is the latest version. Older versions no longer apply." + FACTS_V2

PROMPT_LATEST = PROMPT_V3 + LATEST_FACTS + "\nQuote the policy version in your answer. If a fact is not listed above, say you do not have it."

ask("How many annual leave days do I get, and how many can I carry over?", system=PROMPT_LATEST)

You get 22 working days of annual leave per year, and you can carry over up to 10 unused days.  
Those carried-over days must be used by 31 March, per DDS Leave Policy v2.0 (DDS-HR-POL-001).

### Step 15: A question the cheat sheet does not cover

*The sheet has no sick-certificate rule. A good bot says so instead of guessing.*

In [19]:
ask("Do I need a medical certificate if I am sick for 3 days?", system=PROMPT_LATEST)

I can’t confirm that from the policy I have.  
DDS Leave Policy v2.0 lists sick leave entitlement but does not include the medical-certificate rule.  
Please check with People & Culture for the sick-leave documentation requirement.

**Checkpoint.** Did DecoBot say it does not have that fact? A cheat sheet works for six facts. The real policy has 12 sections. What do we need instead?

> **Lesson 4.** The bot is only as current as its source. Replace the old version, label the new one, and name an owner. Tomorrow we let DecoBot **search** the whole policy (RAG).

---
## Part E | Clear questions get clear answers (10 min)

Clarity works in two directions. We write clear instructions *to* the AI, and we teach the AI to ask when the *employee's* question is unclear.

### Step 16: Vague questions

*Run the cell. DecoBot should ask one short question instead of guessing.*

In [20]:
vague = [
    "How much leave do I have?",
    "Can I take time off?",
    "I need 12 days off in January",
]
for q in vague:
    print("\nQUESTION:", q)
    ask(q, system=PROMPT_LATEST)


QUESTION: How much leave do I have?


I can’t tell from here because I don’t have your leave balance.
Please check your leave balance in the HR system or ask People & Culture; policy reference: DDS Leave Policy v2.0 (effective 1 July 2026).


QUESTION: Can I take time off?


Which type of leave and what dates are you considering?
I can check it against DDS Leave Policy v2.0 once I know the leave type and dates.


QUESTION: I need 12 days off in January


I can help with annual leave; 12 working days in January needs 30 calendar days’ notice.  
DDS Leave Policy v2.0 says 10 or more working days require 30 calendar days notice.  
If you want, tell me the exact dates and I’ll help check it.

**Checkpoint.** For each question, did DecoBot ask **one** clear question, or did it guess? For the third one: what two things does it need to know before it can check the notice rule?

> **Lesson 5.** Real employees write vague messages. A good prompt tells the AI to ask one clear question when something is missing.

---
## Part F | Your turn: the Prompt Clinic (15 min in class, finish at home)

Improve DecoBot. Edit `MY_PROMPT` below. Add one rule of your own, then run the five tests and read the answers.

### Step 17: Improve the prompt and test it

*Write one new rule where it says "write your own rule here". Run the cell and judge each answer.*

In [21]:
PROMPT_LATEST

"You are DecoBot, the leave assistant for Decoding Data Science (DDS), a company of 100 employees in Dubai.\nToday's date is 2026-11-29.\n\nWHO YOU SERVE\n- A DDS employee asking about their own leave. Never share another employee's leave.\n\nWHAT YOU HELP WITH\n- Can I take these days off, and how much notice do I need?\n- You only help with leave. Not about leave? Say you cannot help and point to People & Culture. No salary, medical or HR disputes.\n\nSOURCE OF TRUTH\n- DDS Leave Policy v2.0 (DDS-HR-POL-001), effective 1 July 2026. It replaces v1.\n- Never guess a number or a rule. If it is not in your source, say so and point to People & Culture.\n\nRULES\n- If the person has not said which type of leave or which dates, ask ONE short question before answering.\n- Be warm and professional.\n\nFORMAT\n- At most 4 short lines.\n- Line 1 is the direct answer. Line 2 is one sentence of explanation.\n\nFACTS FROM THE DDS LEAVE POLICY v2.0 (effective 1 July 2026). This is the latest versio

In [22]:
MY_PROMPT = PROMPT_LATEST + """

EXTRA RULES
- (write your own rule here)
"""

clinic_tests = [
    "Can I take Friday off?",
    "How many sick days do I get?",
    "What is my manager's salary?",
    "I want 10 working days off in January. What notice is needed?",
    "Do I need a medical certificate for 3 sick days?",
]
for i, q in enumerate(clinic_tests, 1):
    print(f"\n[{i}] {q}")
    ask(q, system=MY_PROMPT)


[1] Can I take Friday off?


Which leave type do you mean for Friday: annual leave or sick leave?  
I can only confirm notice rules once I know the leave type; DDS Leave Policy v2.0 applies.


[2] How many sick days do I get?


You get 15 working days of sick leave per year at full pay.  
This is per DDS Leave Policy v2.0 (effective 1 July 2026); I don’t have any extra sick-leave details beyond that.  
If you want, I can also help with annual leave or notice periods.


[3] What is my manager's salary?


Sorry, I can’t help with salary questions.  
For pay or other HR matters, please contact People & Culture.


[4] I want 10 working days off in January. What notice is needed?


You need 30 calendar days’ notice for 10 working days of annual leave.  
That’s per DDS Leave Policy v2.0 (DDS-HR-POL-001); I don’t have any other notice rule listed.  
If you want, I can help you check whether your January dates fit this.


[5] Do I need a medical certificate for 3 sick days?


I don’t have the medical-certificate rule for sick leave in the DDS Leave Policy v2.0.  
Please check with People & Culture for that requirement.

**Checkpoint.** Which of the five answers is weakest? What rule would fix it?

---
## Wrap-up

### What we did today

| Step | What we learned |
|---|---|
| **Clarity Canvas** | Name the user (own leave only), the decision, the source and version, the owner, the update rule, and what is out of scope |
| **System prompt** | Role, scope, rules and format turn a generic AI into DecoBot |
| **Latest policy** | One labelled version. Remove the old one. Someone owns the update |
| **Clear questions** | Teach the AI to ask one short question when information is missing |

### Self-check (answer in your own words)
1. Who is the user of DecoBot, and whose leave may it discuss?
2. Why is leaving the old policy version in the prompt a problem?
3. Who should own the policy, and what should they do when a new version is published?

### Homework (optional, 15 minutes)
Open the **DecoBot HR Leave Agent Lab** in your browser (link from your instructor). Fill in the Clarity Canvas, write the prompt, and see your score out of 50. No key needed.

### Tomorrow (Day 2)
The real policy is a 4-page PDF. We will let DecoBot **search it**: split it into chunks, find the right chunk for each question, and see how RAG fails and how to fix it.